# Module 1 — Modern Data Engineering with Snowflake

Welcome to Module 1 of *Introduction to Modern Data Engineering with Snowflake*. This companion notebook lets you follow along with the instructor and run every code cell interactively inside Snowsight.

## Learning objectives
- Understand what "modern data engineering" means on Snowflake and how it differs from legacy warehouse work.
- Stand up a small analytical dataset by transforming public data from the Snowflake Data Cloud (the **Snowflake Public Data (Free)** Marketplace listing).
- Explore the result set with SQL, then visualize it inline with Streamlit — all without leaving the notebook.

## Prerequisites
- Snowflake account with `ACCOUNTADMIN` (or a role able to create databases and warehouses).
- The **Snowflake Public Data (Free)** listing installed from the Marketplace — search **"Finance Economics Snowflake"** in Snowsight → Data Products → Marketplace, then click **Get**. This makes the `SNOWFLAKE_PUBLIC_DATA_FREE` database available in your account.
- Roughly 15–20 minutes.

## What you'll build
Two analytical tables:
- `WAGES_CPI.DATA.ANNUAL_WAGES_CPI_USA` — average annual wages and CPI for the USA, 2012–2022.
- `WAGES_CPI.DATA.MONTHLY_CPI_USA` — average monthly CPI for the USA over the last three years.

Then you'll visualize both inline with Streamlit charts.

## 1. Set your session context

Every notebook starts by pinning the role, warehouse, and database context so every downstream SQL cell has an unambiguous execution environment.

In [ ]:
USE ROLE accountadmin;

CREATE WAREHOUSE IF NOT EXISTS compute_wh;
USE WAREHOUSE compute_wh;

-- Create a database and schema to store our data
CREATE OR REPLACE DATABASE wages_cpi;
CREATE OR REPLACE SCHEMA data;
USE DATABASE wages_cpi;
USE SCHEMA data;

## 2. Build the annual wages + CPI table

This CTE joins two Marketplace sources — OECD annual wages and Bureau of Labor Statistics CPI — and produces a single row per year with both averages side by side.

In [ ]:
-- Creates a table tracking average annual wages and CPI for the USA, between 2012 and 2022
CREATE OR REPLACE TABLE annual_wages_cpi_usa AS
SELECT
  DATE_TRUNC('year', oecd_timeseries.date) AS year,
  ROUND(
    AVG(
      CASE 
        WHEN oecd_attributes.variable_name ILIKE '%annual wages%' THEN oecd_timeseries.value
        ELSE NULL 
      END
    )
  ) AS avg_annual_wages,
  ROUND(
    AVG(
      CASE 
        WHEN bureau_of_labor_statistics_price_attributes.variable_name ILIKE '%CPI%' THEN bureau_of_labor_statistics_price_timeseries.value
        ELSE NULL 
      END
    )
  ) AS cpi
FROM
  SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.OECD_TIMESERIES oecd_timeseries
JOIN 
  SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.OECD_ATTRIBUTES oecd_attributes
  ON oecd_timeseries.variable = oecd_attributes.variable
LEFT JOIN 
  SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_PRICE_TIMESERIES bureau_of_labor_statistics_price_timeseries
  ON DATE_TRUNC('year', oecd_timeseries.date) = DATE_TRUNC('year', bureau_of_labor_statistics_price_timeseries.date)
  AND bureau_of_labor_statistics_price_timeseries.geo_id = 'country/USA'
LEFT JOIN 
  SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_PRICE_ATTRIBUTES bureau_of_labor_statistics_price_attributes
  ON bureau_of_labor_statistics_price_timeseries.variable = bureau_of_labor_statistics_price_attributes.variable
WHERE
  (oecd_attributes.variable_name ILIKE '%annual wages%' 
  OR bureau_of_labor_statistics_price_attributes.variable_name ILIKE '%CPI%')
  AND oecd_timeseries.geo_id = 'country/USA'
  AND DATE_TRUNC('year', oecd_timeseries.date) BETWEEN '2012-01-01' AND '2022-12-31'
GROUP BY
  year
ORDER BY
  year;

Quick sanity check on the annual table:

In [ ]:
SELECT * FROM annual_wages_cpi_usa ORDER BY year;

## 3. Build the monthly CPI table

Now the same shape at monthly grain over the last three years — useful for a higher-resolution view of inflation.

In [ ]:
-- Creates a table tracking CPI for the USA over the last 3 years, on a monthly basis
CREATE OR REPLACE TABLE monthly_cpi_usa AS
SELECT
  DATE_TRUNC('month', bureau_of_labor_statistics_price_timeseries.date) AS month,
  ROUND(
    AVG(
      CASE 
        WHEN bureau_of_labor_statistics_price_attributes.variable_name ILIKE '%CPI%' THEN bureau_of_labor_statistics_price_timeseries.value
        ELSE NULL 
      END
    ), 1
  ) AS avg_cpi
FROM
  SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_PRICE_TIMESERIES bureau_of_labor_statistics_price_timeseries
JOIN 
  SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_PRICE_ATTRIBUTES bureau_of_labor_statistics_price_attributes 
  ON bureau_of_labor_statistics_price_timeseries.variable = bureau_of_labor_statistics_price_attributes.variable
WHERE
  bureau_of_labor_statistics_price_attributes.variable_name ILIKE '%CPI%'
  AND bureau_of_labor_statistics_price_timeseries.geo_id = 'country/USA'
  AND DATE_TRUNC('month', bureau_of_labor_statistics_price_timeseries.date) BETWEEN DATEADD(year, -3, CURRENT_DATE) AND CURRENT_DATE
GROUP BY
  month
ORDER BY
  month;

Quick sanity check on the monthly table:

In [ ]:
SELECT * FROM monthly_cpi_usa ORDER BY month;

## 4. Deploy the results as a Streamlit app

Time to visualize what you built. The repo contains `module-1/wages_cpi_streamlit_app.py` — a Streamlit app that reads the two tables you just created and renders them as charts.

Because your workspace is already connected to this repo via Git, you can create the Streamlit object directly from that file — no file upload, no copy-paste into the Streamlit editor. Run the cell below.

> **Before running:** replace `<YOUR_WORKSPACE_GIT_REPO>` with the fully-qualified name of the Git repository object backing your workspace (for example `my_db.public.snowflake_data_engineering_course`). You can find this in Snowsight under **Data » Databases » your workspace database » Git Repositories**.

In [ ]:
CREATE OR REPLACE STREAMLIT wages_cpi.data.wages_cpi_usa_app
  FROM '@<YOUR_WORKSPACE_GIT_REPO>/branches/main/module-1/'
  MAIN_FILE = 'wages_cpi_streamlit_app.py'
  QUERY_WAREHOUSE = compute_wh
  COMMENT = 'Wages and CPI trends for the USA';

SHOW STREAMLITS IN SCHEMA wages_cpi.data;

### View the app

Two ways to open your new app:

1. **From Snowsight navigation** — go to **Projects » Streamlit** and click `WAGES_CPI_USA_APP`.
2. **Direct URL** — the `SHOW STREAMLITS` result above includes a `url_id` column. The full URL follows the pattern `https://app.snowflake.com/<org>/<account>/#/streamlit-apps/WAGES_CPI.DATA.WAGES_CPI_USA_APP`.

Once it opens, you'll see the two line charts rendered from the tables you just built. Click **Share** in the top-right to grant other roles access or copy a shareable link.

## 5. (Optional) Teardown

Uncomment and run the cell below when you're finished with this module and want to clean up. **Skip this if you plan to move on to Module 2** — later modules build on their own separate database (`tasty_bytes`), so keeping `wages_cpi` around costs almost nothing.

In [ ]:
-- DROP DATABASE IF EXISTS wages_cpi;